# TimesFM


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ml_final_project


In [ ]:
import os
if not os.path.exists('/content/ml_final_project'):
    !git clone -q https://github.com/ochiga07/ml_final_project.git /content/ml_final_project
import sys
sys.path.append('/content/ml_final_project')

from colab_setup import setup_project

drive_repo = setup_project(repo_url="https://github.com/ochiga07/ml_final_project.git")

import feature_pipeline
import importlib
importlib.reload(feature_pipeline)
from feature_pipeline import load_raw_data, run_feature_pipeline

from metrics import wmae


In [ ]:
!pip install -q timesfm mlflow dagshub


In [ ]:
import os
import numpy as np
import pandas as pd
import timesfm
import mlflow
import dagshub
import warnings
warnings.filterwarnings('ignore')

if 'DAGSHUB_USER_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['DAGSHUB_USER_TOKEN'] = userdata.get('DAGSHUB_USER_TOKEN')
    except Exception:
        pass

dagshub.init(repo_owner='aochi23', repo_name='ml_final_project', mlflow=True)
mlflow.set_experiment('TimesFM_Training')


## Data


In [ ]:
train, test, features, stores = load_raw_data(path=f'{drive_repo}/data/')
full_df = run_feature_pipeline(train, test, features, stores)

train_df = full_df[full_df['is_train'] == 1].drop(columns=['is_train'])
print(train_df.shape)


In [ ]:
VAL_WEEKS = 12
MIN_HISTORY = 20

pairs = train_df.groupby(['Store', 'Dept'])
print(f'total pairs: {len(pairs)}')

all_series = []

for (store, dept), grp in pairs:
    grp = grp.sort_values('Date')
    sales = grp['Weekly_Sales'].values
    holidays = grp['IsHoliday'].values
    if len(sales) < MIN_HISTORY + VAL_WEEKS:
        continue
    all_series.append({
        'store': store, 'dept': dept,
        'sales': sales,
        'holidays': holidays,
    })

print(f'pairs with enough history: {len(all_series)}')


## Load model


In [ ]:
import torch
torch.set_float32_matmul_precision('high')

tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
tfm.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=VAL_WEEKS,
        use_continuous_quantile_head=False,
        normalize_inputs=True,
        infer_is_positive=True,
    )
)
print('model loaded (timesfm 2.5 API)')


## Zero-shot eval


In [ ]:
# feed history up to val cutoff, forecast val_weeks
histories = []
actuals = []
hol_flags = []

for s in all_series:
    sales = s['sales']
    hols = s['holidays']
    cutoff = len(sales) - VAL_WEEKS
    histories.append(sales[:cutoff].astype(np.float32))
    actuals.append(sales[cutoff:cutoff + VAL_WEEKS])
    hol_flags.append(hols[cutoff:cutoff + VAL_WEEKS])

print(f'forecasting {len(histories)} series...')


In [ ]:
# run in batches (memory issues)
BATCH = 256
all_forecasts = []

for start in range(0, len(histories), BATCH):
    batch_hist = histories[start:start + BATCH]
    preds, _ = tfm.forecast(horizon=VAL_WEEKS, inputs=batch_hist)
    all_forecasts.append(preds)
    if start % 1024 == 0:
        print(f'  done {start}/{len(histories)}')

forecasts = np.concatenate(all_forecasts, axis=0)
print(f'forecast shape: {forecasts.shape}')


In [ ]:
# calc wmae across pairs
all_pred = forecasts.flatten()
all_true = np.concatenate(actuals)
all_hol = np.concatenate(hol_flags)

score = wmae(all_true, all_pred, all_hol)
print(f'zero-shot val WMAE: {score:.2f}')


In [ ]:
with mlflow.start_run(run_name='TimesFM_ZeroShot'):
    mlflow.log_param('model', 'timesfm-1.0-200m-pytorch')
    mlflow.log_param('horizon_len', VAL_WEEKS)
    mlflow.log_param('n_series', len(all_series))
    mlflow.log_param('freq', 2)
    mlflow.log_metric('val_wmae', score)
    mlflow.log_metric('n_predictions', len(all_pred))
    print(f'logged: wmae={score:.2f}')


## Try different context lengths


In [ ]:
# truncate history to see if shorter/longer context matters
context_lengths = [52, 104, None]  # 1yr, 2yr, full
ctx_results = []

with mlflow.start_run(run_name='TimesFM_ContextLength'):
    for ctx in context_lengths:
        ctx_hists = []
        for h in histories:
            if ctx is not None and len(h) > ctx:
                ctx_hists.append(h[-ctx:])
            else:
                ctx_hists.append(h)

        ctx_forecasts = []
        for start in range(0, len(ctx_hists), BATCH):
            batch = ctx_hists[start:start + BATCH]
            preds, _ = tfm.forecast(horizon=VAL_WEEKS, inputs=batch)
            ctx_forecasts.append(preds)

        fc = np.concatenate(ctx_forecasts, axis=0).flatten()
        s = wmae(all_true, fc, all_hol)

        label = str(ctx) if ctx else 'full'
        mlflow.log_metric(f'wmae_ctx_{label}', s)
        ctx_results.append({'context': label, 'wmae': s})
        print(f'context={label}: wmae={s:.2f}')

ctx_df = pd.DataFrame(ctx_results)
ctx_df


## Summary


In [ ]:
best_ctx = ctx_df.loc[ctx_df['wmae'].idxmin()]
print(f'best context: {best_ctx["context"]}, wmae: {best_ctx["wmae"]:.2f}')
print(f'\ntimesfm is zero-shot so no training needed')
print(f'total series evaluated: {len(all_series)}')
